# 05 — Entrenamiento clasificador de marca (Logo-2K+)

Entrena un EfficientNet-B0 sobre el subset `Clothes` de Logo-2K+
(113 marcas). Aprox. 1-2 horas en T4.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
REPO_DIR = '/content/fashion-feature-extraction'
if not os.path.exists(REPO_DIR):
    SRC = '/content/drive/MyDrive/master_ia/fashion-extraction/code'
    import shutil
    shutil.copytree(SRC, REPO_DIR)
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!pip install -q -r requirements.txt


## 2. Config con paths Drive

In [ ]:
from src.data.colab import setup_colab
config = setup_colab('config/pipeline_config.yaml')
print('Logo-2K+ root:', config['brand']['classes_source'])

from src.brand.dataset import discover_brands
brands = discover_brands(config['brand']['classes_source'])
print(f'{len(brands)} marcas detectadas')
print('Primeras 10:', brands[:10])


## 3. Entrenar

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

from src.brand.train import train_brand

result = train_brand(
    config=config,
    history_path=f"{config['paths']['outputs']}/history_brand.json",
)
print(f"\n✓ Best F1-macro: {result['best_metric']:.3f}")
print(f"✓ Clases: {result['n_classes']}")
print(f"✓ Checkpoint: {result['checkpoint']}")


## 4. Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(result['history'])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history['epoch'], history['train_loss'], 'o-')
axes[0].set_title('Train loss'); axes[0].grid(True)
axes[1].plot(history['epoch'], history['accuracy'], 'o-', label='accuracy')
axes[1].plot(history['epoch'], history['f1_macro'], 'o-', label='F1-macro')
axes[1].set_title('Validation'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()
